## Tinh chỉnh Qwen2.5-Coder-1.5B bằng LoRA trên R2Vul
Nhóm em dùng 5 ngôn ngữ C, C#, Java, JavaScript, Python; giữ nguyên tập chia của tác giả. Đầu vào 2048 token, LoRA r=16, alpha=32, 2 epoch trên Colab T4. Chọn epoch theo AUROC phần C của valid. Đặt ba tệp interim R2Vul `train.jsonl`, `valid.jsonl`, `test.jsonl` trong `Drive của tôi/hocmay_lora/r2vul5/`. Kết quả lưu tại cùng thư mục; có thể chạy tiếp từ checkpoint.


In [ ]:
# Nguyễn Tâm Bình - 2324801030003
DATASET = "r2vul5"
from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/hocmay_lora")
MODEL = "Qwen/Qwen2.5-Coder-1.5B"
CFG = dict(epochs=2)
MAX_LEN, BATCH, ACCUM, LR, SEED, SAVE_EVERY = 2048, 2, 16, 1e-4, 42, 100
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Không có GPU. Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại.")
print("GPU:", torch.cuda.get_device_name(0), "| Bộ:", DATASET)

### Mount Drive, tải dữ liệu, cài thư viện

In [ ]:
# Nguyễn Tâm Bình - 2324801030003
from google.colab import drive
drive.mount("/content/drive")
OUT = DRIVE / DATASET
OUT.mkdir(parents=True, exist_ok=True)
import subprocess, importlib.util
if importlib.util.find_spec("torchao") is not None:
    subprocess.run(["pip", "uninstall", "-y", "-q", "torchao"], check=True)
    importlib.invalidate_caches()
if importlib.util.find_spec("peft") is None:
    subprocess.run(["pip", "install", "-q", "peft"], check=True)
import shutil
from google.colab import files
data_dir = OUT
for ten in ("train.jsonl", "valid.jsonl", "test.jsonl"):
    if not (data_dir / ten).exists():
        print("Chưa có", ten, "trong", data_dir, "- chọn tệp này trên máy để tải lên")
        up = files.upload()
        shutil.move(next(iter(up)), data_dir / ten)
print(sorted(p.name for p in data_dir.iterdir()))

### Chuẩn bị dữ liệu và mô hình

In [ ]:
# Nguyễn Tâm Bình - 2324801030003
import json, math, random, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model
def load(split):
    with open(data_dir / f"{split}.jsonl", encoding="utf-8") as f:
        return [json.loads(line) for line in f]
splits = {s: load(s) for s in ("train", "valid", "test")}
LANG = {r["id"]: r["lang"] for rows in splits.values() for r in rows}
train = splits["train"]
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "right"
def encode(rows):
    ids = tok([r["func"] for r in rows], truncation=True, max_length=MAX_LEN, add_special_tokens=True)["input_ids"]
    return [(r["id"], r["target"], x) for r, x in zip(rows, ids)]
data = {"train": encode(train), "valid": encode(splits["valid"]), "test": encode(splits["test"])}
pos = sum(t for _, t, _ in data["train"])
weights = torch.tensor([len(data["train"]) / (2 * (len(data["train"]) - pos)), len(data["train"]) / (2 * pos)], device="cuda")
print({k: len(v) for k, v in data.items()}, "dương tính train:", pos, "trọng số lớp:", weights.tolist())
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
model.config.pad_token_id = tok.pad_token_id
model.config.use_cache = False
model = model.half().cuda()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
lora = LoraConfig(task_type="SEQ_CLS", r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
model.print_trainable_parameters()

### Huấn luyện (checkpoint mỗi 100 bước, chọn epoch theo AUROC phần C của valid)


In [ ]:
# Nguyễn Tâm Bình - 2324801030003
from sklearn.metrics import roc_auc_score, average_precision_score
def batches(items, seed):
    order = list(range(len(items)))
    random.Random(seed).shuffle(order)
    groups = [order[i:i + BATCH * 50] for i in range(0, len(order), BATCH * 50)]
    out = []
    for g in groups:
        g = sorted(g, key=lambda i: len(items[i][2]))
        out += [g[i:i + BATCH] for i in range(0, len(g), BATCH)]
    random.Random(seed + 1).shuffle(out)
    return out
def collate(items, idx):
    width = max(len(items[i][2]) for i in idx)
    ids = torch.full((len(idx), width), tok.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(idx), width), dtype=torch.long)
    for row, i in enumerate(idx):
        seq = items[i][2]
        ids[row, :len(seq)] = torch.tensor(seq)
        mask[row, :len(seq)] = 1
    return ids.cuda(), mask.cuda(), torch.tensor([items[i][1] for i in idx]).cuda()
@torch.no_grad()
def predict(items):
    model.eval()
    order = sorted(range(len(items)), key=lambda i: len(items[i][2]))
    probs = np.zeros(len(items), dtype=np.float32)
    for k in range(0, len(order), BATCH * 2):
        idx = order[k:k + BATCH * 2]
        ids, mask, _ = collate(items, idx)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(input_ids=ids, attention_mask=mask).logits.float()
        probs[idx] = torch.softmax(logits, -1)[:, 1].cpu().numpy()
    model.train()
    return probs
trainable = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.01)
steps_per_epoch = math.ceil(len(batches(data["train"], SEED)) / ACCUM)
total = steps_per_epoch * CFG["epochs"]
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(1, int(0.05 * total))) * max(0.0, (total - s) / total))
scaler = torch.amp.GradScaler("cuda")
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
state = {"epoch": 0, "step": 0, "best_auroc": -1.0, "history": []}
ck = OUT / "checkpoint.pt"
if ck.exists():
    saved = torch.load(ck, map_location="cuda", weights_only=False)
    model.load_state_dict(saved["model"], strict=False)
    opt.load_state_dict(saved["opt"]); sched.load_state_dict(saved["sched"]); scaler.load_state_dict(saved["scaler"])
    state = saved["state"]
    print("Tiếp tục từ epoch", state["epoch"], "bước", state["step"])
def trainable_state():
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if "lora_" in k or "score" in k}
def save():
    tmp = OUT / "checkpoint.tmp"
    torch.save({"model": trainable_state(), "opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(), "state": state}, tmp)
    tmp.replace(ck)
import time
model.train()
while state["epoch"] < CFG["epochs"]:
    plan = batches(data["train"], SEED + state["epoch"])
    start, t0, running = state["step"] * ACCUM, time.time(), []
    for b in range(start, len(plan)):
        ids, mask, y = collate(data["train"], plan[b])
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(input_ids=ids, attention_mask=mask).logits.float()
        loss = loss_fn(logits, y) / ACCUM
        scaler.scale(loss).backward()
        running.append(loss.item() * ACCUM)
        if (b + 1) % ACCUM == 0 or b + 1 == len(plan):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            state["step"] += 1
            if state["step"] % 20 == 0:
                done = (b + 1 - start) / (time.time() - t0)
                print(f"epoch {state['epoch'] + 1} bước {state['step']}/{steps_per_epoch} loss {np.mean(running[-80:]):.4f} còn khoảng {(len(plan) - b - 1) / done / 60:.0f} phút", flush=True)
            if state["step"] % SAVE_EVERY == 0:
                save()
    probs = predict(data["valid"])
    y = np.array([t for _, t, _ in data["valid"]])
    la_c = np.array([LANG[i] == "c" for i, _, _ in data["valid"]])
    auroc, auprc = roc_auc_score(y[la_c], probs[la_c]), average_precision_score(y[la_c], probs[la_c])
    auroc_all = roc_auc_score(y, probs)
    state["history"].append({"epoch": state["epoch"] + 1, "valid_auroc": float(auroc), "valid_auprc": float(auprc), "valid_auroc_5ngon_ngu": float(auroc_all)})
    print("epoch", state["epoch"] + 1, "valid C AUROC", round(auroc, 4), "AUPRC", round(auprc, 4), "| cả 5 ngôn ngữ AUROC", round(auroc_all, 4), flush=True)
    if auroc > state["best_auroc"]:
        state["best_auroc"] = float(auroc)
        torch.save(trainable_state(), OUT / "best_adapter.pt")
    state["epoch"] += 1
    state["step"] = 0
    save()
print("Xong huấn luyện:", state["history"])

### Dự đoán valid, test bằng epoch tốt nhất và ghi kết quả lên Drive

In [ ]:
# Nguyễn Tâm Bình - 2324801030003
import csv
from sklearn.metrics import f1_score
model.load_state_dict(torch.load(OUT / "best_adapter.pt", map_location="cuda"), strict=False)
for split in ("valid", "test"):
    path = OUT / f"pred_{split}.csv"
    if path.exists():
        continue
    probs = predict(data[split])
    tmp = OUT / f"pred_{split}.tmp"
    with open(tmp, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["id", "lang", "target", "prob"])
        for (i, t, _), p in zip(data[split], probs):
            w.writerow([i, LANG[i], t, f"{p:.6f}"])
    tmp.replace(path)
    y = np.array([t for _, t, _ in data[split]])
    la = np.array([LANG[i] for i, _, _ in data[split]])
    for ng in ("c", "c_sharp", "java", "javascript", "python"):
        m = la == ng
        print(split, ng, "AUROC", round(roc_auc_score(y[m], probs[m]), 4), "F1@0.5", round(f1_score(y[m], probs[m] >= 0.5), 4))
info = {"dataset": DATASET, "model": MODEL, "lora": {"r": 16, "alpha": 32, "dropout": 0.05, "modules": "q,k,v,o,gate,up,down"}, "max_len": MAX_LEN, "batch": BATCH * ACCUM, "lr": LR, "seed": SEED, "train_size": len(data["train"]), "class_weights": weights.tolist(), "history": state["history"]}
(OUT / "info.json").write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
(OUT / "XONG").touch()
ck.unlink(missing_ok=True)
print("Đã ghi", sorted(p.name for p in OUT.iterdir()))